In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

input_path = Path(r"C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed\train_full_missingness_audit_v1.csv")

df = pd.read_csv(input_path)

print(df.shape)
df.head()

(80000, 105)


,patient_id,site_id,triage_nurse_id,arrival_mode,arrival_hour,arrival_day,arrival_month,arrival_season,shift,age,...,diastolic_bp_missing_flag,respiratory_rate_missing_flag,temperature_missing_flag,map_missing_flag,pulse_pressure_missing_flag,shock_index_missing_flag,pain_not_recorded_flag,any_vital_missing_flag,any_triage_data_missing_flag,missingness_pattern_category
0,TG-UXRGA9UCO,SITE-TMP-01,NURSE-0033,walk-in,6,Monday,5,spring,morning,43,...,0,0,0,0,0,0,0,0,0,00_no_relevant_missing
1,TG-B19DBBS2G,SITE-HEL-01,NURSE-0001,walk-in,6,Thursday,4,spring,morning,72,...,0,0,0,0,0,0,1,0,1,07_pain_only_not_recorded
2,TG-GZ97W7M6V,SITE-HEL-02,NURSE-0005,walk-in,8,Saturday,4,spring,morning,82,...,0,0,0,0,0,0,0,0,0,00_no_relevant_missing
3,TG-THIB2TN9Q,SITE-HEL-02,NURSE-0026,police,7,Sunday,3,spring,morning,50,...,0,0,0,0,0,0,0,0,0,00_no_relevant_missing
4,TG-J3U3LQ2QY,SITE-HEL-02,NURSE-0044,walk-in,5,Tuesday,5,spring,night,62,...,0,0,0,0,0,0,0,0,0,00_no_relevant_missing


In [2]:
df["disposition"].value_counts(dropna=False)

disposition
discharged     39028
admitted       24601
transferred     5203
observation     4337
lwbs            3656
lama            2764
deceased         411
Name: count, dtype: int64

In [3]:
disposition_summary = (
    df["disposition"]
    .value_counts(dropna=False)
    .rename_axis("disposition")
    .reset_index(name="n")
)

disposition_summary["%"] = (
    disposition_summary["n"] / len(df) * 100
).round(2)

disposition_summary

,disposition,n,%
0,discharged,39028,48.78
1,admitted,24601,30.75
2,transferred,5203,6.50
3,observation,4337,5.42
4,lwbs,3656,4.57
5,lama,2764,3.45
6,deceased,411,0.51


In [4]:
pd.crosstab(
    df["triage_acuity"],
    df["disposition"]
)

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
triage_acuity,,,,,,,
1,2312,267,67,0,0,98,478
2,8754,144,1641,125,128,1070,1577
3,11049,0,10868,1116,1179,2358,2351
4,2267,0,16609,1202,1566,693,683
5,219,0,9843,321,783,118,114


In [5]:
disposition_by_esi_pct = (
    pd.crosstab(
        df["triage_acuity"],
        df["disposition"],
        normalize="index"
    ) * 100
).round(2)

disposition_by_esi_pct

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
triage_acuity,,,,,,,
1,71.76,8.29,2.08,0.00,0.00,3.04,14.84
2,65.14,1.07,12.21,0.93,0.95,7.96,11.73
3,38.20,0.00,37.58,3.86,4.08,8.15,8.13
4,9.85,0.00,72.15,5.22,6.80,3.01,2.97
5,1.92,0.00,86.36,2.82,6.87,1.04,1.00


In [6]:
disposition_by_esi_pct = (
    pd.crosstab(
        df["triage_acuity"],
        df["disposition"],
        normalize="index"
    ) * 100
).round(2)

disposition_by_esi_pct

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
triage_acuity,,,,,,,
1,71.76,8.29,2.08,0.00,0.00,3.04,14.84
2,65.14,1.07,12.21,0.93,0.95,7.96,11.73
3,38.20,0.00,37.58,3.86,4.08,8.15,8.13
4,9.85,0.00,72.15,5.22,6.80,3.01,2.97
5,1.92,0.00,86.36,2.82,6.87,1.04,1.00


In [7]:
esi_by_disposition_pct = (
    pd.crosstab(
        df["disposition"],
        df["triage_acuity"],
        normalize="index"
    ) * 100
).round(2)

esi_by_disposition_pct

triage_acuity,1,2,3,4,5
disposition,,,,,
admitted,9.40,35.58,44.91,9.22,0.89
deceased,64.96,35.04,0.00,0.00,0.00
discharged,0.17,4.20,27.85,42.56,25.22
lama,0.00,4.52,40.38,43.49,11.61
lwbs,0.00,3.50,32.25,42.83,21.42
observation,2.26,24.67,54.37,15.98,2.72
transferred,9.19,30.31,45.19,13.13,2.19


In [8]:
serious_strict_values = ["admitted", "transferred", "deceased"]

df["outcome_serious_strict"] = df["disposition"].isin(
    serious_strict_values
).astype(int)

In [9]:
serious_plus_observation_values = [
    "admitted",
    "transferred",
    "deceased",
    "observation"
]

df["outcome_serious_plus_observation"] = df["disposition"].isin(
    serious_plus_observation_values
).astype(int)

In [10]:
incomplete_care_values = ["lwbs", "lama"]

df["outcome_incomplete_care"] = df["disposition"].isin(
    incomplete_care_values
).astype(int)

In [11]:
df["outcome_deceased"] = (df["disposition"] == "deceased").astype(int)

In [12]:
outcome_cols = [
    "outcome_serious_strict",
    "outcome_serious_plus_observation",
    "outcome_incomplete_care",
    "outcome_deceased"
]

outcome_summary = pd.DataFrame({
    "outcome": outcome_cols,
    "n_positive": [df[col].sum() for col in outcome_cols],
    "pct_positive": [(df[col].mean() * 100).round(2) for col in outcome_cols]
})

outcome_summary

,outcome,n_positive,pct_positive
0,outcome_serious_strict,30215,37.77
1,outcome_serious_plus_observation,34552,43.19
2,outcome_incomplete_care,6420,8.02
3,outcome_deceased,411,0.51


In [13]:
outcomes_by_esi_n = (
    df.groupby("triage_acuity")[outcome_cols]
    .sum()
)

outcomes_by_esi_n

,outcome_serious_strict,outcome_serious_plus_observation,outcome_incomplete_care,outcome_deceased
triage_acuity,,,,
1,3057,3155,0,267
2,10475,11545,253,144
3,13400,15758,2295,0
4,2950,3643,2768,0
5,333,451,1104,0


In [14]:
outcomes_by_esi_pct = (
    df.groupby("triage_acuity")[outcome_cols]
    .mean() * 100
).round(2)

outcomes_by_esi_pct

,outcome_serious_strict,outcome_serious_plus_observation,outcome_incomplete_care,outcome_deceased
triage_acuity,,,,
1,94.88,97.92,0.00,8.29
2,77.94,85.91,1.88,1.07
3,46.33,54.49,7.94,0.00
4,12.81,15.83,12.02,0.00
5,2.92,3.96,9.69,0.00


In [15]:
esi_3_5_serious = df[
    (df["triage_acuity"].isin([3, 4, 5])) &
    (df["outcome_serious_strict"] == 1)
]

print(f"ESI 3-5 con serious strict: {len(esi_3_5_serious)}")
print(f"% del dataset total: {len(esi_3_5_serious) / len(df) * 100:.2f}%")

ESI 3-5 con serious strict: 16683
% del dataset total: 20.85%


In [16]:
esi_3_5_serious["triage_acuity"].value_counts().sort_index()

triage_acuity
3    13400
4     2950
5      333
Name: count, dtype: int64

In [17]:
esi_3_5_serious["disposition"].value_counts()

disposition
admitted       13535
transferred     3148
Name: count, dtype: int64

In [18]:
esi_4_5_serious = df[
    (df["triage_acuity"].isin([4, 5])) &
    (df["outcome_serious_strict"] == 1)
]

print(f"ESI 4-5 con serious strict: {len(esi_4_5_serious)}")
print(f"% del dataset total: {len(esi_4_5_serious) / len(df) * 100:.2f}%")

ESI 4-5 con serious strict: 3283
% del dataset total: 4.10%


In [19]:
esi_4_5_serious["triage_acuity"].value_counts().sort_index()

triage_acuity
4    2950
5     333
Name: count, dtype: int64

In [20]:
esi_4_5_serious["disposition"].value_counts()

disposition
admitted       2486
transferred     797
Name: count, dtype: int64

In [21]:
df["candidate_hidden_risk_esi_3_5"] = (
    df["triage_acuity"].isin([3, 4, 5]) &
    (df["outcome_serious_strict"] == 1)
).astype(int)

df["candidate_hidden_risk_esi_4_5"] = (
    df["triage_acuity"].isin([4, 5]) &
    (df["outcome_serious_strict"] == 1)
).astype(int)

In [22]:
hidden_risk_summary = pd.DataFrame({
    "candidate_group": [
        "candidate_hidden_risk_esi_3_5",
        "candidate_hidden_risk_esi_4_5"
    ],
    "n": [
        df["candidate_hidden_risk_esi_3_5"].sum(),
        df["candidate_hidden_risk_esi_4_5"].sum()
    ],
    "%": [
        (df["candidate_hidden_risk_esi_3_5"].mean() * 100).round(2),
        (df["candidate_hidden_risk_esi_4_5"].mean() * 100).round(2)
    ]
})

hidden_risk_summary

,candidate_group,n,%
0,candidate_hidden_risk_esi_3_5,16683,20.85
1,candidate_hidden_risk_esi_4_5,3283,4.10


In [23]:
outcome_comparison_by_esi = (
    df.groupby("triage_acuity")[
        ["outcome_serious_strict", "outcome_serious_plus_observation"]
    ]
    .mean() * 100
).round(2)

outcome_comparison_by_esi

,outcome_serious_strict,outcome_serious_plus_observation
triage_acuity,,
1,94.88,97.92
2,77.94,85.91
3,46.33,54.49
4,12.81,15.83
5,2.92,3.96


In [24]:
outcome_comparison_by_esi["difference_plus_obs_minus_strict"] = (
    outcome_comparison_by_esi["outcome_serious_plus_observation"] -
    outcome_comparison_by_esi["outcome_serious_strict"]
).round(2)

outcome_comparison_by_esi

,outcome_serious_strict,outcome_serious_plus_observation,difference_plus_obs_minus_strict
triage_acuity,,,
1,94.88,97.92,3.04
2,77.94,85.91,7.97
3,46.33,54.49,8.16
4,12.81,15.83,3.02
5,2.92,3.96,1.04


In [25]:
observation_by_esi = pd.crosstab(
    df["triage_acuity"],
    df["disposition"].eq("observation"),
    normalize="index"
) * 100

observation_by_esi.round(2)

disposition,False,True
triage_acuity,,
1,96.96,3.04
2,92.04,7.96
3,91.85,8.15
4,96.99,3.01
5,98.96,1.04


In [26]:
clinical_cols = [
    "age",
    "heart_rate",
    "respiratory_rate",
    "temperature_c",
    "spo2",
    "gcs_total",
    "pain_score",
    "news2_score",
    "num_comorbidities",
    "num_active_medications"
]

summary_by_serious = (
    df.groupby("outcome_serious_strict")[clinical_cols]
    .median()
    .round(2)
)

summary_by_serious

,age,heart_rate,respiratory_rate,temperature_c,spo2,gcs_total,pain_score,news2_score,num_comorbidities,num_active_medications
outcome_serious_strict,,,,,,,,,,
0,48.0,85.5,16.5,37.3,97.8,15.0,4.0,1.0,5.0,4.0
1,48.0,98.3,19.5,38.0,94.8,15.0,7.0,5.0,5.0,4.0


In [27]:
summary_by_serious_plus_obs = (
    df.groupby("outcome_serious_plus_observation")[clinical_cols]
    .median()
    .round(2)
)

summary_by_serious_plus_obs

,age,heart_rate,respiratory_rate,temperature_c,spo2,gcs_total,pain_score,news2_score,num_comorbidities,num_active_medications
outcome_serious_plus_observation,,,,,,,,,,
0,48.0,84.8,16.3,37.3,97.9,15.0,3.0,1.0,5.0,4.0
1,48.0,97.8,19.3,38.0,95.0,15.0,7.0,4.0,5.0,4.0


In [28]:
if "pp_category_label" in df.columns:
    pp_outcome_summary = (
        df.groupby("pp_category_label")[outcome_cols]
        .mean() * 100
    ).round(2)
    
    pp_outcome_summary

In [29]:
if "missingness_pattern_category" in df.columns:
    missingness_outcome_summary = (
        df.groupby("missingness_pattern_category")[outcome_cols]
        .mean() * 100
    ).round(2)
    
    missingness_outcome_summary

In [30]:
if "anthro_quality_category" in df.columns:
    anthro_outcome_summary = (
        df.groupby("anthro_quality_category")[outcome_cols]
        .mean() * 100
    ).round(2)
    
    anthro_outcome_summary

In [31]:
incomplete_by_esi = (
    df.groupby("triage_acuity")["outcome_incomplete_care"]
    .mean() * 100
).round(2)

incomplete_by_esi

triage_acuity
1     0.00
2     1.88
3     7.94
4    12.02
5     9.69
Name: outcome_incomplete_care, dtype: float64

In [32]:
pd.crosstab(
    df["triage_acuity"],
    df["disposition"]
)[["lwbs", "lama"]]

disposition,lwbs,lama
triage_acuity,,
1,0,0
2,128,125
3,1179,1116
4,1566,1202
5,783,321


In [33]:
conditions = [
    df["disposition"].isin(["admitted", "transferred", "deceased"]),
    df["disposition"].eq("observation"),
    df["disposition"].isin(["lwbs", "lama"]),
    df["disposition"].eq("discharged")
]

choices = [
    "01_serious_strict",
    "02_observation_intermediate",
    "03_incomplete_care",
    "00_discharged"
]

df["outcome_group_clinical"] = np.select(
    conditions,
    choices,
    default="99_unknown"
)

In [34]:
outcome_group_summary = (
    df["outcome_group_clinical"]
    .value_counts(dropna=False)
    .rename_axis("outcome_group_clinical")
    .reset_index(name="n")
)

outcome_group_summary["%"] = (
    outcome_group_summary["n"] / len(df) * 100
).round(2)

outcome_group_summary

,outcome_group_clinical,n,%
0,00_discharged,39028,48.78
1,01_serious_strict,30215,37.77
2,03_incomplete_care,6420,8.02
3,02_observation_intermediate,4337,5.42


In [35]:
pd.crosstab(
    df["triage_acuity"],
    df["outcome_group_clinical"]
)

outcome_group_clinical,00_discharged,01_serious_strict,02_observation_intermediate,03_incomplete_care
triage_acuity,,,,
1,67,3057,98,0
2,1641,10475,1070,253
3,10868,13400,2358,2295
4,16609,2950,693,2768
5,9843,333,118,1104


In [36]:
(
    pd.crosstab(
        df["triage_acuity"],
        df["outcome_group_clinical"],
        normalize="index"
    ) * 100
).round(2)

outcome_group_clinical,00_discharged,01_serious_strict,02_observation_intermediate,03_incomplete_care
triage_acuity,,,,
1,2.08,94.88,3.04,0.00
2,12.21,77.94,7.96,1.88
3,37.58,46.33,8.15,7.94
4,72.15,12.81,3.01,12.02
5,86.36,2.92,1.04,9.69


In [37]:
output_dir = Path(r"C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "train_full_outcomes_v1.csv"

df.to_csv(output_path, index=False)

print(f"Archivo guardado en: {output_path}")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Archivo guardado en: C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed\train_full_outcomes_v1.csv
Filas: 80000
Columnas: 112


In [38]:
df_check = pd.read_csv(output_path)

print(df_check.shape)

df_check[
    [
        "disposition",
        "triage_acuity",
        "outcome_serious_strict",
        "outcome_serious_plus_observation",
        "outcome_incomplete_care",
        "candidate_hidden_risk_esi_3_5",
        "candidate_hidden_risk_esi_4_5",
        "outcome_group_clinical"
    ]
].head()

(80000, 112)


,disposition,triage_acuity,outcome_serious_strict,outcome_serious_plus_observation,outcome_incomplete_care,candidate_hidden_risk_esi_3_5,candidate_hidden_risk_esi_4_5,outcome_group_clinical
0,discharged,2,0,0,0,0,0,00_discharged
1,discharged,5,0,0,0,0,0,00_discharged
2,discharged,5,0,0,0,0,0,00_discharged
3,discharged,3,0,0,0,0,0,00_discharged
4,transferred,3,1,1,0,1,0,01_serious_strict


# Conclusiones definición de outcome

## Objetivo

Definir desenlaces candidatos para entrenar una capa de seguridad clínica en triage. El objetivo del proyecto no es predecir el ESI asignado ni predecir muerte de forma aislada, sino estimar riesgo clínico posterior usando información disponible al momento del triage y luego comparar esa señal de riesgo con el ESI para detectar posibles discordancias.

## Outcome principal

Se definió como outcome principal:

`outcome_serious_strict`

Donde:

- 1 = `admitted`, `transferred`, `deceased`
- 0 = `discharged`, `observation`, `lwbs`, `lama`

Este outcome representa una aproximación pragmática a disposición clínicamente seria posterior al triage. No se interpreta como desenlace clínico puro, ya que hospitalización y traslado también pueden depender de disponibilidad, protocolos y contexto institucional. Sin embargo, es una señal posterior relevante y disponible para entrenar un modelo de riesgo.

## Outcome secundario

Se definió como outcome secundario:

`outcome_serious_plus_observation`

Donde:

- 1 = `admitted`, `transferred`, `deceased`, `observation`
- 0 = `discharged`, `lwbs`, `lama`

La categoría `observation` se considera intermedia. Puede reflejar incertidumbre clínica, necesidad de vigilancia o decisiones de flujo asistencial. Por esta razón no se incluyó en el outcome principal, pero sí se mantuvo como análisis de sensibilidad.

## Outcome exploratorio operacional

Se definió como outcome exploratorio:

`outcome_incomplete_care`

Donde:

- 1 = `lwbs`, `lama`
- 0 = todas las demás disposiciones

LWBS y LAMA no se interpretan automáticamente como desenlaces graves ni benignos. Representan atención incompleta o interrumpida, por lo que se analizarán por separado como dimensión operacional de seguridad.

## Hallazgos principales

El outcome serio estricto muestra un gradiente claro por ESI, con mayor frecuencia en ESI 1-2 y menor frecuencia en ESI 4-5. Esto sugiere que el ESI se relaciona con el desenlace posterior, pero también existen pacientes ESI 3-5 y ESI 4-5 con hospitalización o traslado posterior.

El grupo ESI 4-5 con outcome serio estricto no contiene fallecidos en este dataset, pero sí contiene pacientes hospitalizados o trasladados. Por lo tanto, el objetivo no será afirmar detección de muertes subtriageadas, sino detectar riesgo clínico serio posterior y posibles discordancias entre riesgo predicho y acuidad asignada.

## Decisión metodológica

Para el modelo principal:

- Usar `outcome_serious_strict` como target.
- No usar `triage_acuity` como predictor principal.
- Usar `triage_acuity` como referencia, baseline y variable de análisis de discordancia.
- No usar `disposition` ni `ed_los_hours` como predictores.
- Evaluar el modelo especialmente en subgrupos ESI 3-5 y ESI 4-5.

## Enfoque del proyecto

El proyecto se orientará como una capa de seguridad para triage, no como reemplazo del ESI. El modelo estimará riesgo de disposición seria posterior usando datos disponibles al triage. Luego, el riesgo predicho será comparado con el ESI asignado para identificar pacientes cuyo riesgo estimado parece mayor que su clasificación inicial.